<a href="https://colab.research.google.com/github/Lokuvithana/LLM-Multi-Layer-Security-Framework/blob/rp%2F2026%2Fc2/models/C2_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

convert .parquet file formats to CSV format

In [ ]:
from datasets import load_dataset

ds = load_dataset("medalpaca/medical_meadow_medqa")

In [ ]:
print(ds)
print(ds[list(ds.keys())[0]].column_names)
print(ds[list(ds.keys())[0]][0])

DatasetDict({
    train: Dataset({
        features: ['input', 'instruction', 'output'],
        num_rows: 10178
    })
})
['input', 'instruction', 'output']
{'input': "Q:A 23-year-old pregnant woman at 22 weeks gestation presents with burning upon urination. She states it started 1 day ago and has been worsening despite drinking more water and taking cranberry extract. She otherwise feels well and is followed by a doctor for her pregnancy. Her temperature is 97.7°F (36.5°C), blood pressure is 122/77 mmHg, pulse is 80/min, respirations are 19/min, and oxygen saturation is 98% on room air. Physical exam is notable for an absence of costovertebral angle tenderness and a gravid uterus. Which of the following is the best treatment for this patient?? \n{'A': 'Ampicillin', 'B': 'Ceftriaxone', 'C': 'Ciprofloxacin', 'D': 'Doxycycline', 'E': 'Nitrofurantoin'},", 'instruction': 'Please answer with one of the option in the bracket', 'output': 'E: Nitrofurantoin'}


In [ ]:
import os

output_dir = "./medical_meadow_medqa"

os.makedirs(output_dir, exist_ok=True)

for split in ds:
    ds[split].to_csv(f"{output_dir}/{split}.csv")

print(f"Dataset saved to: {os.path.abspath(output_dir)}")

Creating CSV from Arrow format:   0%|          | 0/11 [00:00<?, ?ba/s]

Dataset saved to: /content/medical_meadow_medqa


In [ ]:
from datasets import load_dataset

ds = load_dataset("deepset/prompt-injections")

README.md:   0%|          | 0.00/500 [00:00<?, ?B/s]

data/train-00000-of-00001-9564e8b05b4757(…): reconstructing file:   0%|          |  0.00B / 40.3kB            

data/train-00000-of-00001-9564e8b05b4757(…): downloading bytes:           |  0.00B            

data/test-00000-of-00001-701d16158af8736(…): reconstructing file:   0%|          |  0.00B / 10.9kB            

data/test-00000-of-00001-701d16158af8736(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/546 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/116 [00:00<?, ? examples/s]

In [ ]:
print(ds)
print(ds[list(ds.keys())[0]].column_names)
print(ds[list(ds.keys())[0]][0])

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 546
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 116
    })
})
['text', 'label']
{'text': 'Refugee crisis in Europe solutions', 'label': 0}


In [ ]:
from datasets import load_dataset
import os

ds = load_dataset("deepset/prompt-injections")

output_dir = "./prompt-injections"
os.makedirs(output_dir, exist_ok=True)

for split in ds:
    ds[split].to_csv(f"{output_dir}/{split}.csv", index=False)

print(f"Dataset saved to: {os.path.abspath(output_dir)}")

Creating CSV from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating CSV from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Dataset saved to: /content/prompt-injections


In [ ]:
import os
print(os.listdir("./prompt-injections"))

['train.csv', 'test.csv']


In [ ]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("HFXM/CARES-18K")

README.md:   0%|          | 0.00/2.46k [00:00<?, ?B/s]

data/train-00000-of-00001-b38d4c3a00be5b(…): reconstructing file:   0%|          |  0.00B / 2.48MB            

data/train-00000-of-00001-b38d4c3a00be5b(…): downloading bytes:           |  0.00B            

data/test-00000-of-00001-48ff445f6c5e20b(…): reconstructing file:   0%|          |  0.00B / 2.48MB            

data/test-00000-of-00001-48ff445f6c5e20b(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/9239 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/9239 [00:00<?, ? examples/s]

In [ ]:
from datasets import load_dataset
import os

# Load the dataset (requires Hugging Face access if gated)
ds = load_dataset("HFXM/CARES-18K")

# Create a local directory
output_dir = "./CARES-18K"
os.makedirs(output_dir, exist_ok=True)

# Save each split as a CSV file
for split in ds:
    ds[split].to_csv(
        os.path.join(output_dir, f"{split}.csv"),
        index=False
    )

print(f"Dataset saved to: {os.path.abspath(output_dir)}")

Creating CSV from Arrow format:   0%|          | 0/10 [00:00<?, ?ba/s]

Creating CSV from Arrow format:   0%|          | 0/10 [00:00<?, ?ba/s]

Dataset saved to: /content/CARES-18K


In [ ]:
import os
print(os.listdir("./CARES-18K"))

['train.csv', 'test.csv']


Mount Drive & imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import re
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

BASE = "/content/drive/MyDrive/LLM-C2"
RANDOM_STATE = 42
RELEVANT_PRINCIPLES = [0, 6]       # Patient Privacy & Consent, AI/Clinical-Safety-Boundary
OBFUSCATE_CAP_RATIO = 0.25         # cap on obfuscate-method rows within the attack class

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


CARES: pool, filter, relabel, cap obfuscate

In [ ]:
cares = pd.concat([
    pd.read_csv(f"{BASE}/CARES-18K/carestrain.csv"),
    pd.read_csv(f"{BASE}/CARES-18K/carestest.csv"),
], ignore_index=True)

cares = cares.drop_duplicates(subset=["prompt"])  # the two files are near-duplicates
cares = cares[cares["principle_index"].isin(RELEVANT_PRINCIPLES)].copy()
cares["label"] = (cares["harmful_level"] > 0).astype(int)
cares["text"] = cares["prompt"]
cares["group"] = cares["base_prompt"]       # keeps every variant of a prompt on one side of the split
cares["source"] = "cares-18k"

attack_obf = cares[(cares.label == 1) & (cares.method == "obfuscate")]
attack_other = cares[(cares.label == 1) & (cares.method != "obfuscate")]
max_obf = int(len(attack_other) * OBFUSCATE_CAP_RATIO / (1 - OBFUSCATE_CAP_RATIO)) if len(attack_other) else 0
if len(attack_obf) > max_obf:
    attack_obf = attack_obf.sample(n=max_obf, random_state=RANDOM_STATE)

cares = pd.concat([cares[cares.label == 0], attack_other, attack_obf], ignore_index=True)
print(f"CARES: {len(cares)} rows ({(cares.label==1).sum()} attack, {(cares.label==0).sum()} benign)")

CARES: 4691 rows (3532 attack, 1159 benign)


deepset/prompt-injections: English only

In [ ]:
deepset = pd.concat([
    pd.read_csv(f"{BASE}/prompt-injections/prompttrain.csv"),
    pd.read_csv(f"{BASE}/prompt-injections/prompttest.csv"),
], ignore_index=True)

def looks_german(t: str) -> bool:
    if re.search(r"[äöüßÄÖÜ]", t):
        return True
    return bool(re.search(r"\b(und|der|die|das|ist|nicht|für|mit|ich|sie|wir|auf|eine)\b", t.lower()))

deepset = deepset[~deepset["text"].apply(looks_german)].copy()
deepset["group"] = deepset.index.astype(str) + "-deepset"
deepset["source"] = "deepset-prompt-injections"
print(f"deepset: {len(deepset)} rows ({(deepset.label==1).sum()} attack, {(deepset.label==0).sum()} benign)")

deepset: 416 rows (160 attack, 256 benign)


medical_meadow_medqa: the actual imbalance fix

In [ ]:
medical = pd.read_csv(f"{BASE}/medical_meadow_medqa/medicaltrain.csv")
medical["text"] = medical["input"].str.split(r"\n\{").str[0].str.strip()
medical = medical[medical["text"].str.len() > 20]
medical["label"] = 0
medical["group"] = medical.index.astype(str) + "-medqa"
medical["source"] = "medical_meadow_medqa"
print(f"medical_meadow_medqa available: {len(medical)} rows")

# THE FIX: without this, all 10,178 rows go in as benign and swamp the attack
# class 3-to-1. Sample down so total benign ≈ 1.15x total attack, not 3x+.
attack_count = (cares.label == 1).sum() + (deepset.label == 1).sum()
other_benign_count = (cares.label == 0).sum() + (deepset.label == 0).sum()
medical_target = max(0, int(attack_count * 1.15) - other_benign_count)
medical_target = min(medical_target, len(medical))
medical = medical.sample(n=medical_target, random_state=RANDOM_STATE)
print(f"medical_meadow_medqa sampled down to: {len(medical)} rows")

medical_meadow_medqa available: 10178 rows
medical_meadow_medqa sampled down to: 2830 rows


Combine, group-safe split, save back to Drive

In [ ]:
cols = ["text", "label", "source", "group"]
combined = pd.concat([cares[cols], deepset[cols], medical[cols]], ignore_index=True)
combined = combined.drop_duplicates(subset=["text"])
print(f"\nCombined corpus: {len(combined)} rows "
      f"({(combined.label==1).sum()} attack, {(combined.label==0).sum()} benign, "
      f"{(combined.label==1).mean()*100:.1f}% attack)")

gss = GroupShuffleSplit(n_splits=1, train_size=0.875, random_state=RANDOM_STATE)
train_idx, val_idx = next(gss.split(combined, groups=combined["group"]))
train_df = combined.iloc[train_idx].reset_index(drop=True)
val_df = combined.iloc[val_idx].reset_index(drop=True)

train_df.to_csv(f"{BASE}/hc_attackcorpus_train.csv", index=False)
val_df.to_csv(f"{BASE}/hc_attackcorpus_val.csv", index=False)

print(f"\nTrain: {len(train_df)} rows ({(train_df.label==1).sum()} attack, {(train_df.label==0).sum()} benign)")
print(f"Val:   {len(val_df)} rows ({(val_df.label==1).sum()} attack, {(val_df.label==0).sum()} benign)")
print(f"\nSaved to {BASE}/hc_attackcorpus_train.csv and hc_attackcorpus_val.csv")


Combined corpus: 7937 rows (3692 attack, 4245 benign, 46.5% attack)

Train: 6951 rows (3265 attack, 3686 benign)
Val:   986 rows (427 attack, 559 benign)

Saved to /content/drive/MyDrive/LLM-C2/hc_attackcorpus_train.csv and hc_attackcorpus_val.csv


Reserve the frozen test pool BEFORE train/val (this is the fix)

In [ ]:
# Previously: 100% of filtered data went into train+val, leaving nothing for
# real evaluation. Fix: carve out 20% of GROUPS first, untouched by anything
# that follows, THEN split the remaining 80% into train/val.
gss_outer = GroupShuffleSplit(n_splits=1, train_size=0.80, random_state=RANDOM_STATE)
model_idx, frozen_idx = next(gss_outer.split(combined, groups=combined["group"]))
modeling_pool = combined.iloc[model_idx].reset_index(drop=True)
frozen_pool = combined.iloc[frozen_idx].reset_index(drop=True)

print(f"Modeling pool: {len(modeling_pool)} rows | Frozen test pool: {len(frozen_pool)} rows")

gss_inner = GroupShuffleSplit(n_splits=1, train_size=0.875, random_state=RANDOM_STATE)
train_idx, val_idx = next(gss_inner.split(modeling_pool, groups=modeling_pool["group"]))
train_df = modeling_pool.iloc[train_idx].reset_index(drop=True)
val_df = modeling_pool.iloc[val_idx].reset_index(drop=True)

train_df.to_csv(f"{BASE}/hc_attackcorpus_train.csv", index=False)
val_df.to_csv(f"{BASE}/hc_attackcorpus_val.csv", index=False)
print(f"Train: {len(train_df)} ({(train_df.label==1).sum()} attack, {(train_df.label==0).sum()} benign)")
print(f"Val:   {len(val_df)} ({(val_df.label==1).sum()} attack, {(val_df.label==0).sum()} benign)")

Modeling pool: 6359 rows | Frozen test pool: 1578 rows
Train: 5552 (2645 attack, 2907 benign)
Val:   807 (361 attack, 446 benign)


Build the four frozen test sets from that reserved pool

In [ ]:
# deepset and medical have no 'method' column natively (only CARES does) —
# add a placeholder so it survives the merge
if "method" not in deepset.columns:
    deepset["method"] = "n/a"
if "method" not in medical.columns:
    medical["method"] = "n/a"

cols = ["text", "label", "source", "group", "method"]
combined = pd.concat([cares[cols], deepset[cols], medical[cols]], ignore_index=True)
combined = combined.drop_duplicates(subset=["text"])
print(f"Combined corpus: {len(combined)} rows "
      f"({(combined.label==1).sum()} attack, {(combined.label==0).sum()} benign)")

Combined corpus: 7937 rows (3692 attack, 4245 benign)


In [ ]:
if "method" not in deepset.columns:
    deepset["method"] = "n/a"
if "method" not in medical.columns:
    medical["method"] = "n/a"

cols = ["text", "label", "source", "group", "method"]
combined = pd.concat([cares[cols], deepset[cols], medical[cols]], ignore_index=True)
combined = combined.drop_duplicates(subset=["text"])

In [ ]:
gss_outer = GroupShuffleSplit(n_splits=1, train_size=0.80, random_state=RANDOM_STATE)
model_idx, frozen_idx = next(gss_outer.split(combined, groups=combined["group"]))
modeling_pool = combined.iloc[model_idx].reset_index(drop=True)
frozen_pool = combined.iloc[frozen_idx].reset_index(drop=True)

gss_inner = GroupShuffleSplit(n_splits=1, train_size=0.875, random_state=RANDOM_STATE)
train_idx, val_idx = next(gss_inner.split(modeling_pool, groups=modeling_pool["group"]))
train_df = modeling_pool.iloc[train_idx].reset_index(drop=True)
val_df = modeling_pool.iloc[val_idx].reset_index(drop=True)

train_df.to_csv(f"{BASE}/hc_attackcorpus_train.csv", index=False)
val_df.to_csv(f"{BASE}/hc_attackcorpus_val.csv", index=False)

In [ ]:
import re

BLOCKLIST_PATTERN = re.compile(
    r"ignore (all |any )?(previous|prior|above)|disregard (the )?(above|previous)|"
    r"you are now|forget (your|all) (instructions|rules)", re.IGNORECASE)

benign_test = frozen_pool[frozen_pool.label == 0]
attacks = frozen_pool[frozen_pool.label == 1]

known_attacks_test = attacks[attacks.method == "direct"]
adversarial_paraphrase_test = attacks[attacks.method.isin(["indirect", "obfuscate", "role_play"])]

c1_residual_candidates = attacks[attacks.method.isin(["indirect", "role_play"])]
c1_residual_test = c1_residual_candidates[
    ~c1_residual_candidates.text.apply(lambda t: bool(BLOCKLIST_PATTERN.search(t)))
]

for name, df_ in [("known_attacks_test", known_attacks_test),
                   ("adversarial_paraphrase_test", adversarial_paraphrase_test),
                   ("c1_residual_test", c1_residual_test),
                   ("benign_test", benign_test)]:
    df_.to_csv(f"{BASE}/{name}.csv", index=False)
    print(f"{name}: {len(df_)} rows saved")

known_attacks_test: 198 rows saved
adversarial_paraphrase_test: 455 rows saved
c1_residual_test: 290 rows saved
benign_test: 892 rows saved


In [ ]:
!pip install -q transformers datasets evaluate accelerate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.7 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from datasets import Dataset

BASE = "/content/drive/MyDrive/LLM-C2"
train_df = pd.read_csv(f"{BASE}/hc_attackcorpus_train.csv")
val_df = pd.read_csv(f"{BASE}/hc_attackcorpus_val.csv")

train_ds = Dataset.from_pandas(train_df[["text", "label"]])
val_ds = Dataset.from_pandas(val_df[["text", "label"]])
print(train_ds, val_ds)

Dataset({
    features: ['text', 'label'],
    num_rows: 5552
}) Dataset({
    features: ['text', 'label'],
    num_rows: 807
})


In [ ]:
import re

def clean_text(t):
    t = t.strip()
    t = re.sub(r'\n+', ' ', t)       # collapse newlines (found in 4 rows)
    t = re.sub(r'\s+', ' ', t)        # collapse repeated whitespace
    return t

train_df["text"] = train_df["text"].apply(clean_text)
val_df["text"] = val_df["text"].apply(clean_text)

# Drop degenerate/near-empty examples (found 1: "ukraina", 7 chars)
MIN_LENGTH = 15
train_df = train_df[train_df["text"].str.len() >= MIN_LENGTH].reset_index(drop=True)
val_df = val_df[val_df["text"].str.len() >= MIN_LENGTH].reset_index(drop=True)

print(f"Train after cleaning: {len(train_df)} rows")
print(f"Val after cleaning: {len(val_df)} rows")

Train after cleaning: 5547 rows
Val after cleaning: 807 rows


tokenize the cleaned data

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

train_ds = Dataset.from_pandas(train_df[["text", "label"]])
val_ds = Dataset.from_pandas(val_df[["text", "label"]])

MODEL_NAME = "google/electra-small-discriminator"
MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=MAX_LENGTH)

train_tok = train_ds.map(tokenize, batched=True)
val_tok = val_ds.map(tokenize, batched=True)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/5547 [00:00<?, ? examples/s]

Map:   0%|          | 0/807 [00:00<?, ? examples/s]

Load the model

In [ ]:
MODEL_NAME = "google/electra-small-discriminator"

from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 54.2MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 54.2MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: google/electra-small-discriminator
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
classifier.dense.bias                             | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstre

Define metrics

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

Training arguments + start training

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd, re, numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

BASE = "/content/drive/MyDrive/LLM-C2"
MODEL_NAME = "google/electra-small-discriminator"
MAX_LENGTH = 256

train_df = pd.read_csv(f"{BASE}/hc_attackcorpus_train.csv")
val_df = pd.read_csv(f"{BASE}/hc_attackcorpus_val.csv")

def clean_text(t):
    t = re.sub(r'\s+', ' ', t.strip())
    return t
train_df["text"] = train_df["text"].apply(clean_text)
val_df["text"] = val_df["text"].apply(clean_text)
train_df = train_df[train_df["text"].str.len() >= 15].reset_index(drop=True)
val_df = val_df[val_df["text"].str.len() >= 15].reset_index(drop=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=MAX_LENGTH)
train_tok = Dataset.from_pandas(train_df[["text", "label"]]).map(tokenize, batched=True)
val_tok = Dataset.from_pandas(val_df[["text", "label"]]).map(tokenize, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {"accuracy": accuracy_score(labels, preds), "precision": precision, "recall": recall, "f1": f1}

args = TrainingArguments(
    output_dir=f"{BASE}/c2_classifier_checkpoints",
    eval_strategy="epoch", save_strategy="epoch",
    learning_rate=3e-5, per_device_train_batch_size=16, per_device_eval_batch_size=32,
    num_train_epochs=4, weight_decay=0.01,
    load_best_model_at_end=True, metric_for_best_model="f1",
    logging_steps=50, report_to="none",
)

trainer = Trainer(model=model, args=args, train_dataset=train_tok, eval_dataset=val_tok,
                   compute_metrics=compute_metrics)
trainer.train()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/5547 [00:00<?, ? examples/s]

...

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1388, training_loss=0.15335120703026608, metrics={'train_runtime': 12295.8112, 'train_samples_per_second': 1.805, 'train_steps_per_second': 0.113, 'total_flos': 326381495390208.0, 'train_loss': 0.15335120703026608, 'epoch': 4.0})

find your saved checkpoints

In [2]:
BASE = "/content/drive/MyDrive/LLM-C2"
!ls {BASE}/c2_classifier_checkpoints

checkpoint-1041  checkpoint-1388  checkpoint-347  checkpoint-694


load the model directly from that checkpoint (no retraining)

In [3]:
import re, torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "google/electra-small-discriminator"
MAX_LENGTH = 256
BEST_CHECKPOINT = f"{BASE}/c2_classifier_checkpoints/checkpoint-1388"  # adjust if ls shows a different highest number

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)   # tokenizer is unchanged by training, safe to reload fresh
model = AutoModelForSequenceClassification.from_pretrained(BEST_CHECKPOINT)
model.eval()

def clean_text(t):
    return re.sub(r'\s+', ' ', t.strip())

print("Loaded trained model from checkpoint — ready to evaluate, no retraining needed.")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

Loaded trained model from checkpoint — ready to evaluate, no retraining needed.


now save a clean, clearly-named final copy

In [4]:
model.save_pretrained(f"{BASE}/c2_classifier_final")
tokenizer.save_pretrained(f"{BASE}/c2_classifier_final")
print("Saved to c2_classifier_final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to c2_classifier_final


now run the frozen-test-set evaluation

In [5]:
def evaluate_test_set(path, expected_label):
    df = pd.read_csv(f"{BASE}/{path}")
    df["text"] = df["text"].apply(clean_text)
    enc = tokenizer(list(df["text"]), truncation=True, padding=True, max_length=MAX_LENGTH, return_tensors="pt")
    with torch.no_grad():
        logits = model(**enc).logits
    preds = torch.argmax(logits, dim=-1).numpy()
    probs = torch.softmax(logits, dim=-1)[:, 1].numpy()

    if expected_label == 1:
        recall = (preds == 1).mean()
        print(f"{path}: recall = {recall:.3f}  (n={len(df)}, mean P(attack)={probs.mean():.3f})")
    else:
        fpr = (preds == 1).mean()
        print(f"{path}: false-positive rate = {fpr:.3f}  (n={len(df)}, mean P(attack)={probs.mean():.3f})")

print("=== Classifier-only performance on frozen test sets ===")
evaluate_test_set("known_attacks_test.csv", expected_label=1)
evaluate_test_set("adversarial_paraphrase_test.csv", expected_label=1)
evaluate_test_set("c1_residual_test.csv", expected_label=1)
evaluate_test_set("benign_test.csv", expected_label=0)

=== Classifier-only performance on frozen test sets ===
known_attacks_test.csv: recall = 0.990  (n=198, mean P(attack)=0.981)
adversarial_paraphrase_test.csv: recall = 0.976  (n=455, mean P(attack)=0.967)
c1_residual_test.csv: recall = 0.983  (n=290, mean P(attack)=0.973)
benign_test.csv: false-positive rate = 0.048  (n=892, mean P(attack)=0.050)


In [6]:
df = pd.read_csv(f"{BASE}/adversarial_paraphrase_test.csv")
df["text"] = df["text"].apply(clean_text)
enc = tokenizer(list(df["text"]), truncation=True, padding=True, max_length=MAX_LENGTH, return_tensors="pt")
with torch.no_grad():
    logits = model(**enc).logits
preds = torch.argmax(logits, dim=-1).numpy()
df["correct"] = (preds == 1)

print(df.groupby("method")["correct"].agg(["mean", "count"]))

               mean  count
method                    
indirect   0.981132    159
obfuscate  0.963636    165
role_play  0.984733    131
